# Stage 3: PCA & MLP-Phase Baseline (B3)
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection

In [ ]:
!pip install kagglehub -q

In [ ]:
import numpy as np, pandas as pd, os, glob, json
from typing import Dict, Tuple, List
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi']=120; sns.set_style('whitegrid')

INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  Dataset: {INPUT_DIR}')

CONFIG = {'n_pca':128, 'hidden':[256,128,64], 'drop':0.3, 'lr':1e-3, 'wd':1e-4, 'bs':64, 'epochs':50, 'patience':10}

In [ ]:
# Cell 2: Load Phase Maps
cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, 'phase_maps_*.npy')))
if not cache_files: raise FileNotFoundError(f'No cache in {CACHE_DIR}. Run Stage 2!')
phase_results = np.load(cache_files[-1], allow_pickle=True).item()

meta_files = glob.glob(os.path.join(INPUT_DIR, '**', 'metadata.csv'), recursive=True)
metadata_df = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf))) for mf in meta_files], ignore_index=True) if meta_files else None

X_list, y_list = [], []
for gen,maps in phase_results.items():
    if metadata_df is not None:
        gdf=metadata_df[metadata_df['generator']==gen]; is_real=len(gdf)>0 and gdf['target'].mode().iloc[0]==0
    else: is_real='real' in gen.lower()
    for m in maps: X_list.append(m.flatten()); y_list.append(0 if is_real else 1)
X_raw=np.array(X_list,dtype=np.float32); y=np.array(y_list,dtype=np.int64)
print(f'Features: {X_raw.shape}  Real:{(y==0).sum()} Fake:{(y==1).sum()}')

In [ ]:
# Cell 3: PCA
scaler=StandardScaler(); X_scaled=scaler.fit_transform(X_raw)
n_max=min(X_scaled.shape[0],X_scaled.shape[1],500)
pca_full=PCA(n_components=n_max,random_state=42); pca_full.fit(X_scaled)
cum_var=np.cumsum(pca_full.explained_variance_ratio_)
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
a1.bar(range(1,min(51,n_max+1)),pca_full.explained_variance_ratio_[:50],color='steelblue')
a1.set_xlabel('Component'); a1.set_title('Individual Variance')
a2.plot(range(1,len(cum_var)+1),cum_var,lw=2); a2.axhline(0.95,color='r',ls='--',label='95%')
nc=CONFIG['n_pca']
if nc<=len(cum_var): a2.axvline(nc,color='g',ls=':',label=f'n={nc} ({cum_var[nc-1]:.1%})')
a2.legend(); a2.set_xlim(0,min(300,n_max)); plt.tight_layout(); plt.show()
pca=PCA(n_components=nc,random_state=42); X_pca=pca.fit_transform(X_scaled)
print(f'PCA: {X_pca.shape}, var={pca.explained_variance_ratio_.sum():.3f}')
np.save(os.path.join(MODEL_DIR,'pca_components.npy'),pca.components_)

In [ ]:
# Cell 4: MLP Training
class MLPPhase(nn.Module):
    def __init__(self,inp,hids,drop=0.3):
        super().__init__()
        layers=[]; prev=inp
        for h in hids: layers.extend([nn.Linear(prev,h),nn.BatchNorm1d(h),nn.ReLU(True),nn.Dropout(drop)]); prev=h
        layers.append(nn.Linear(prev,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(-1)

X_tv,X_te,y_tv,y_te=train_test_split(X_pca,y,test_size=0.2,stratify=y,random_state=42)
X_tr,X_va,y_tr,y_va=train_test_split(X_tv,y_tv,test_size=0.15,stratify=y_tv,random_state=42)
print(f'Train:{len(X_tr)} Val:{len(X_va)} Test:{len(X_te)}')
trl=DataLoader(TensorDataset(torch.FloatTensor(X_tr),torch.LongTensor(y_tr)),batch_size=64,shuffle=True)
val=DataLoader(TensorDataset(torch.FloatTensor(X_va),torch.LongTensor(y_va)),batch_size=64)
tel=DataLoader(TensorDataset(torch.FloatTensor(X_te),torch.LongTensor(y_te)),batch_size=64)

model=MLPPhase(nc,CONFIG['hidden'],CONFIG['drop']).to(DEVICE)
n_params=sum(p.numel() for p in model.parameters()); print(f'MLP params: {n_params:,}')
crit=nn.BCEWithLogitsLoss(); opt=optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
sched=optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=5)
hist={'tl':[],'vl':[],'ta':[],'va':[]}
best,pat=0.0,0
for ep in tqdm(range(CONFIG['epochs']),desc='MLP'):
    model.train(); tl,tp,tt=0,[],[]
    for xb,yb in trl:
        xb,yb=xb.to(DEVICE),yb.float().to(DEVICE); opt.zero_grad()
        lo=model(xb); l=crit(lo,yb); l.backward(); opt.step()
        tl+=l.item()*len(yb); tp.append(torch.sigmoid(lo).detach().cpu().numpy()); tt.append(yb.cpu().numpy())
    tp,tt=np.concatenate(tp),np.concatenate(tt)
    ta=roc_auc_score(tt,tp) if len(np.unique(tt))>1 else 0
    model.eval(); vl,vp,vt=0,[],[]
    with torch.no_grad():
        for xb,yb in val:
            xb,yb=xb.to(DEVICE),yb.float().to(DEVICE); lo=model(xb)
            vl+=crit(lo,yb).item()*len(yb); vp.append(torch.sigmoid(lo).cpu().numpy()); vt.append(yb.cpu().numpy())
    vp,vt=np.concatenate(vp),np.concatenate(vt)
    va2=roc_auc_score(vt,vp) if len(np.unique(vt))>1 else 0
    sched.step(va2)
    hist['tl'].append(tl/len(trl.dataset)); hist['vl'].append(vl/len(val.dataset)); hist['ta'].append(ta); hist['va'].append(va2)
    if va2>best: best=va2;pat=0;torch.save(model.state_dict(),os.path.join(MODEL_DIR,'mlp_b3_best.pth'))
    else:
        pat+=1
        if pat>=CONFIG['patience']: print(f'Early stop ep {ep+1}'); break
model.load_state_dict(torch.load(os.path.join(MODEL_DIR,'mlp_b3_best.pth'),weights_only=True))

In [ ]:
# Cell 5: Evaluation
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,5))
ep=range(1,len(hist['tl'])+1)
a1.plot(ep,hist['tl'],label='Train');a1.plot(ep,hist['vl'],label='Val');a1.set_title('Loss');a1.legend()
a2.plot(ep,hist['ta'],label='Train');a2.plot(ep,hist['va'],label='Val');a2.set_title('AUC');a2.legend()
plt.tight_layout();plt.show()

model.eval();ap,at=[],[]
with torch.no_grad():
    for xb,yb in tel: ap.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy());at.append(yb.numpy())
tp,tt=np.concatenate(ap),np.concatenate(at)
tacc=accuracy_score(tt,(tp>0.5).astype(int)); tauc=roc_auc_score(tt,tp)
print(f'MLP-Phase(B3): Acc={tacc:.4f} AUC={tauc:.4f} Params={n_params:,}')
print(classification_report(tt,(tp>0.5).astype(int),target_names=['Real','Fake']))

fig,(a1,a2)=plt.subplots(1,2,figsize=(12,5))
fpr,tpr,_=roc_curve(tt,tp); a1.plot(fpr,tpr,lw=2,label=f'AUC={tauc:.3f}');a1.plot([0,1],[0,1],'k--',alpha=0.3);a1.legend();a1.set_title('ROC')
cm=confusion_matrix(tt,(tp>0.5).astype(int))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=a2,xticklabels=['Real','Fake'],yticklabels=['Real','Fake']);a2.set_title('Confusion')
plt.tight_layout();plt.show()
with open(os.path.join(MODEL_DIR,'results_b3.json'),'w') as f:
    json.dump({'model':'MLP-Phase (B3)','test_accuracy':float(tacc),'test_auc':float(tauc),'n_parameters':n_params},f,indent=2)
print('B3 saved.')